# Introduction to pandas
Each section matches a slide. Run the cells in order.

## From a list of dictionaries to a DataFrame

In [ ]:
import pandas as pd

rows = [
    {"reporter": "Kenya", "year": 2023,
     "exports": 7412.5},
    {"reporter": "Ghana", "year": 2023,
     "exports": 16800.0},
]
small = pd.DataFrame(rows)
small

## Loading the trade data

In [ ]:
trade = pd.read_csv(
    "../../data/trade_summary.csv")
print(trade.shape)
trade[["reporter", "year",
       "exports_usd_m"]].head(3)

## info

In [ ]:
trade.info()

## One column is a Series

In [ ]:
exports = trade["exports_usd_m"]
print(type(exports))
print(len(exports))
exports.head(3)

## Series statistics

In [ ]:
print(round(exports.sum()))
print(round(exports.mean(), 1))
print(exports.max())
print(exports.min())
print(trade["reporter"].nunique())

## describe

In [ ]:
exports.describe().round(1)

## Selecting several columns

In [ ]:
cols = ["reporter", "year", "exports_usd_m"]
subset = trade[cols]
print(subset.shape)
subset.head(3)

## Selecting by position with iloc

In [ ]:
# rows 0 to 2, columns 0 to 2
trade.iloc[0:3, 0:3]

## Selecting by label with loc

In [ ]:
trade.loc[0:2, ["reporter", "region"]]

## A comparison makes a mask

In [ ]:
mask = trade["region"] == "Africa"
print(mask.head())
print(mask.sum(), "rows are True")

## Using the mask

In [ ]:
africa = trade[mask]
print(len(africa))
africa[["reporter", "region"]].head(3)

## Combining conditions

In [ ]:
recent_agr = trade[
    (trade["year"] >= 2022)
    & (trade["product_code"] == "AGR")
]
print(len(recent_agr))
recent_agr[["reporter", "year"]].head(3)

## query

In [ ]:
big = trade.query("exports_usd_m > 500000")
print(len(big))
big[["reporter", "year"]].head(3)

## Sorting

In [ ]:
top = (
    trade[trade["year"] == 2023]
    .sort_values("exports_usd_m",
                 ascending=False)
    .head(5)
)
top[["reporter", "exports_usd_m"]]

## Calculated columns

In [ ]:
trade["balance_usd_m"] = (
    trade["exports_usd_m"]
    - trade["imports_usd_m"]
)
trade["exports_usd_bn"] = (
    trade["exports_usd_m"] / 1000).round(1)
trade[["reporter", "balance_usd_m",
       "exports_usd_bn"]].head(3)

## True and False columns

In [ ]:
trade["surplus"] = trade["balance_usd_m"] > 0
print(trade["surplus"].sum(), "surpluses")
trade[["reporter", "balance_usd_m",
       "surplus"]].head(3)

## groupby with one column

In [ ]:
by_region = (
    trade.groupby("region")["exports_usd_m"]
    .sum()
    .round(0)
)
by_region

## groupby with two columns

In [ ]:
by_region_year = (
    trade.groupby(["region", "year"])
    ["exports_usd_m"]
    .sum()
    .reset_index()
)
by_region_year.head()

## agg

In [ ]:
summary = trade.groupby("reporter").agg(
    total=("exports_usd_m", "sum"),
    years=("year", "nunique"),
)
summary.sort_values("total",
                    ascending=False).head()

## pivot_table

In [ ]:
pivot = trade.pivot_table(
    index="region", columns="year",
    values="exports_usd_m", aggfunc="sum")
# USD billions, whole numbers
(pivot / 1000).round().astype(int)

## pct_change

In [ ]:
totals = trade.pivot_table(
    index="reporter", columns="year",
    values="exports_usd_m", aggfunc="sum")
growth = totals.pct_change(axis=1) * 100
growth.round(1).head(6)

## merge preview

In [ ]:
countries = pd.read_excel(
    "../../data/countries.xlsx")
merged = trade.merge(
    countries[["iso3", "income_group"]],
    left_on="reporter_iso3",
    right_on="iso3", how="left")
merged.groupby("income_group")[
    "exports_usd_m"].sum()